[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Реализуйте **Multi-Head Attention (MHA)** с нуля. Несколько голов позволяют модели одновременно учить разные проекционные подпространства и разные схемы связи между позициями; это не простое копирование одной и той же attention matrix.

Каждая из четырёх Linear-проекций обучаема. `W_q/W_k/W_v` сначала переводят весь `d_model`, после чего признаки логически делятся на `H` голов размера `d_k=d_model/H`. После независимого attention головы конкатенируются, а `W_o` снова смешивает информацию между ними. Обязательно `d_model % num_heads == 0`.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

## Поток форм

Для `Q=(B,S_q,D)` и `K,V=(B,S_k,D)` проекции сохраняют последнюю размерность D. После reshape+transpose Q имеет `(B,H,S_q,d_k)`, K/V — `(B,H,S_k,d_k)`. Scores имеют `(B,H,S_q,S_k)`, softmax идёт по `S_k`, head outputs — `(B,H,S_q,d_k)`, concat — `(B,S_q,D)`.

Пример: при `D=128`, `H=8`, `S_q=3`, `S_k=10` размер головы 16, а scores — `(B,8,3,10)`. Это cross-attention: запросы и источник имеют разную длину, что не мешает головам.

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

## Практический план

- Сохраните `num_heads` и `d_k`, создайте четыре Linear-модуля.
- Используйте отдельные длины `S_q` и `S_k`; helper reshape не должен предполагать self-attention.
- После attention верните sequence перед heads: transpose, затем `contiguous()`/reshape.
- Только после concat примените output projection.

## Быстрые проверки

- Self-attention и cross-attention возвращают `(B,S_q,D)`.
- Каждая строка весов каждой головы суммируется в 1 по `S_k`.
- `H=1` остаётся корректным частным случаем.
- Backward достигает всех четырёх проекций и входов.

## Частые ошибки

- Использовать длину Q при reshape K/V.
- Масштабировать на `sqrt(d_model)` вместо `sqrt(d_k)`.
- Softmax по оси голов или запросов.
- Слить память после transpose через несовместимый `view` без contiguous layout.

## Материалы для глубокого изучения

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — раздел 3.2.2 о multi-head attention.
- [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) — промышленный интерфейс и формы.
- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html) — реализация MHA/GQA из базовых операций.

## Вопросы для самопроверки

1. Почему масштаб зависит от размера одной головы?
2. На каком шаге информация разных голов снова смешивается?
3. Какие оси отличаются между self- и cross-attention?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, Q, K, V):
        pass  # Implement multi-head attention

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("mha")